In [9]:
!pip install selenium requests beautifulsoup4 pandas tqdm lxml

In [20]:
import re
import time
import html
import pandas as pd
from tqdm import tqdm
from bs4 import BeautifulSoup
from urllib.parse import quote
from pathlib import Path
from dataclasses import dataclass, asdict

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

BASE_URL = "https://www.law.go.kr"

# 정규식들
RE_ARTICLE = re.compile(r"(제\d+조(?:의\d+)?)\s*(?:\(([^)]*)\))?")
RE_CLAUSE_SPLIT = re.compile(r'(①|②|③|④|⑤|⑥|⑦|⑧|⑨|⑩|⑪|⑫|⑬|⑭|⑮|⑯|⑰|⑱|⑲|⑳)')
CIRCLED = "①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮⑯⑰⑱⑲⑳"
CIRCLED_MAP = {
    "①": "1항", "②": "2항", "③": "3항", "④": "4항", "⑤": "5항",
    "⑥": "6항", "⑦": "7항", "⑧": "8항", "⑨": "9항", "⑩": "10항",
    "⑪": "11항", "⑫": "12항", "⑬": "13항", "⑭": "14항", "⑮": "15항",
    "⑯": "16항", "⑰": "17항", "⑱": "18항", "⑲": "19항", "⑳": "20항"
}

def convert_clause_mark(mark: str) -> str:
    """① → 1항 같은 형식으로 변환. mark가 None 또는 빈 문자열이면 그대로 반환."""
    if not mark:
        return ""
    return CIRCLED_MAP.get(mark, mark)  # 매핑 없으면 그대로 반환

def clean_spaces(s: str) -> str:
    return re.sub(r'\s+', ' ', s).strip()

def extract_text_flow(tag):
    return clean_spaces(" ".join(tag.stripped_strings))


In [9]:
def build_detail_url(law_name: str) -> str:
    encoded = quote(law_name, encoding="utf-8")
    return f"https://www.law.go.kr/법령/{encoded}"

In [10]:
def get_law_iframe_rendered_html(law_name: str, timeout_sec: int = 15):
    url = build_detail_url(law_name)

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1600,1400")

    driver = webdriver.Chrome(options=options)

    try:
        driver.get(url)
        wait = WebDriverWait(driver, timeout_sec)

        # iframe 진입
        iframe_el = wait.until(EC.presence_of_element_located((By.ID, "lawService")))
        driver.switch_to.frame(iframe_el)

        # 본문 로딩되는 요소들
        selectors = ["#conScroll", "#lsBody", "div.law_view", "div.viewer", "#contentBody"]
        for sel in selectors:
            try:
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, sel)))
                break
            except:
                pass

        time.sleep(1)
        return driver.page_source

    finally:
        driver.quit()


In [11]:
def split_clauses(text: str):
    text = clean_spaces(text)

    # '생략' 포함 → 쪼개지 않음
    if "생략" in text:
        return [(None, text)]

    # 항 번호 없음
    if not RE_CLAUSE_SPLIT.search(text):
        return [(None, text)]

    result = []
    matches = list(RE_CLAUSE_SPLIT.finditer(text))

    for i, m in enumerate(matches):
        mark = m.group(1)
        start = m.end()
        end = matches[i+1].start() if i+1 < len(matches) else len(text)
        chunk = text[start:end].strip()
        if chunk:
            result.append((mark, chunk))

    return result or [(None, text)]


def build_full_text(clause_list):
    return clean_spaces(" ".join([t for _, t in clause_list if t]))

In [21]:
@dataclass
class LawRow:
    law_name: str
    section: str         # 장 이름 or "부칙"
    chapter: str         # 전체 제목
    article_number: str
    article_title: str
    clause_number: str   # 전문, ①,②,…
    text: str


def parse_article_pgroup(pgroup, law_name, section, chapter):
    rows = []

    header_p = pgroup.find("p", class_="pty1_p4")
    if not header_p:
        return rows

    # 헤더: 제12조(○○○)
    label = header_p.find("label")
    header_text = label.get_text() if label else header_p.get_text()

    m = RE_ARTICLE.match(clean_spaces(header_text))
    if not m:
        return rows

    article_number = m.group(1)
    article_title = m.group(2) or ""

    # 헤더 후 본문
    header_body = clean_spaces(header_p.get_text().replace(header_text, "", 1))

    # 나머지 문단
    body = []
    for p in pgroup.find_all("p"):
        if p is header_p:
            continue
        txt = clean_spaces(p.get_text(" ", strip=True))
        if txt:
            body.append(txt)

    full_text = clean_spaces(" ".join([header_body] + body))

    clause_list = split_clauses(full_text)
    # full_clause_text = build_full_text(clause_list)

    # # 전문 행
    # rows.append(LawRow(law_name, section, chapter,
    #                    article_number, article_title,
    #                    "전문", full_clause_text))

    # 각 항
    for mark, txt in clause_list:
        converted = convert_clause_mark(mark)
        rows.append(LawRow(
            law_name, section, chapter,
            article_number, article_title,
            converted, txt
    ))
    return rows

In [22]:
def parse_buchik_pgroup(pgroup, law_name):
    rows = []
    section = "부칙"

    head = pgroup.find("p", class_="pty3")
    if not head:
        return rows

    sfon = head.find_all("span", class_="sfon")

    # 법률 제xxxx호 + (지방자치법)
    law_info = html.unescape(sfon[0].get_text()).strip("<>")
    law_ref = html.unescape(sfon[1].get_text()) if len(sfon) > 1 else ""
    chapter = clean_spaces(f"{law_info} {law_ref}")

    # 실제 조문들
    for p in pgroup.find_all("p", class_=re.compile(r"pty3_dep\d+")):
        raw = clean_spaces(p.get_text(" ", strip=True))
        if not raw:
            continue

        bl = p.find("span", class_="bl")
        if bl:
            header = clean_spaces(bl.get_text())
            m = RE_ARTICLE.match(header)
            if m:
                article_number = m.group(1)
                article_title = m.group(2) or ""

                # 본문 추출
                after = p.get_text().replace(bl.get_text(), "", 1)
                body = clean_spaces(after)
            else:
                continue
        else:
            # '제2조부터 제21조까지 생략'
            m = RE_ARTICLE.match(raw)
            if m:
                article_number = m.group(1)
                article_title = m.group(2) or ""
                body = raw
            else:
                continue

        clause_list = split_clauses(body)
        full_clause_text = build_full_text(clause_list)

        # 전문
        rows.append(LawRow(law_name, section, chapter,
                           article_number, article_title,
                           "전문", full_clause_text))

        # 각 항
        for mark, txt in clause_list:
            rows.append(LawRow(law_name, section, chapter,
                               article_number, article_title,
                               mark or "", txt))

    return rows

In [23]:
def parse_law_page(html_source):
    soup = BeautifulSoup(html_source, "html.parser")

    law_name_tag = soup.select_one("#lawName a") or soup.select_one("#lawName")
    law_name = clean_spaces(law_name_tag.get_text()) if law_name_tag else ""

    rows = []
    current_section = "본칙"   # 기본은 본칙
    current_chapter = ""

    for pg in soup.find_all("div", class_="pgroup"):
        # 부칙인지 검사
        if pg.find("p", class_="pty3"):
            rows.extend(parse_buchik_pgroup(pg, law_name))
            continue

        # 🔹 장 제목 찾기: 기존 pty1_tit + 새 구조 gtit 둘 다 지원
        title_tag = pg.find("p", class_=re.compile(r"pty1_tit"))
        if not title_tag:
            # 여기만 추가
            title_tag = pg.find("p", class_="gtit")

        if title_tag:
            current_chapter = clean_spaces(title_tag.get_text())
            # 부칙이 아닌 경우 section은 항상 "본칙"으로 고정
            current_section = "본칙"

        # 조문 파싱
        rows.extend(parse_article_pgroup(
            pg,
            law_name,
            current_section,
            current_chapter
        ))

    return rows

In [24]:
def crawl_law(law_name):
    html = get_law_iframe_rendered_html(law_name)
    rows = parse_law_page(html)

    # 🔧 law_name이 비어 있어도, 호출할 때 넘긴 이름으로 강제 세팅
    for r in rows:
        r.law_name = law_name

    return rows

In [25]:
OUTPUT_DIR = Path("./laws_csv")
OUTPUT_DIR.mkdir(exist_ok=True)

def save_single_law_csv(law_name):
    rows = crawl_law(law_name)
    df = pd.DataFrame([asdict(r) for r in rows])
    df.to_csv(OUTPUT_DIR / f"{law_name}.csv", encoding="utf-8-sig", index=False)
    return df


def save_all_merged(law_names, merged_path="./ALL_LAWS.csv"):
    all_rows = []

    for name in tqdm(law_names):
        df = save_single_law_csv(name)
        all_rows.append(df)

    merged_df = pd.concat(all_rows, ignore_index=True)
    merged_df.to_csv(merged_path, encoding="utf-8-sig", index=False)
    return merged_df

In [27]:
# 예: 엑셀에서 불러온 법령명 리스트
df_list = pd.read_excel("샌드박스_법령명_163개_리스트.xlsx")
law_names = df_list["정식 법령명"].dropna().tolist()

merged_df = save_all_merged(law_names)
merged_df.head()

100%|████████████████████████████████████████████████████████████████████████████████| 163/163 [23:16<00:00,  8.56s/it]


,law_name,section,chapter,article_number,article_title,clause_number,text
0,간선급행버스체계의 건설 및 운영에 관한 특별법,본칙,제1장 총칙,제1조,목적,,이 법은 대도시권등의 교통 문제를 해결하기 위하여 간선급행버스체계의 건설 및 운영에...
1,간선급행버스체계의 건설 및 운영에 관한 특별법,본칙,제1장 총칙,제2조,정의,,이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2022. 6. 10.> 1....
2,간선급행버스체계의 건설 및 운영에 관한 특별법,본칙,제1장 총칙,제3조,다른 법률과의 관계,,간선급행버스체계의 건설 및 운영에 관하여는 다른 법률에 특별한 규정이 있는 경우를 ...
3,간선급행버스체계의 건설 및 운영에 관한 특별법,본칙,제1장 총칙,제4조,간선급행버스체계 종합계획의 수립 등,1항,국토교통부장관은 효율적인 간선급행버스체계를 건설하기 위하여 5년 단위로 간선급행버스...
4,간선급행버스체계의 건설 및 운영에 관한 특별법,본칙,제1장 총칙,제4조,간선급행버스체계 종합계획의 수립 등,2항,종합계획에는 다음 각 호의 사항이 포함되어야 한다. 1. 간선급행버스체계의 중장기 ...


In [5]:
os.getcwd()

'C:\\work\\03. 제안서\\2025\\AGI\\데이터'